In [1]:
import os
import torch
import math
import time
import inspect
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from dataclasses import dataclass
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tokenizers import Tokenizer
from tqdm.notebook import tqdm

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
tokenize = Tokenizer.from_file("./tokenizer/tokenizer.json")

In [4]:
@dataclass
class CFG:
    max_seq_len: int = 1024
    vocab_size: int = tokenize.get_vocab_size()
    dim: int = 768
    n_layers: int = 12
    n_heads_q: int = 12
    n_heads_kv: Optional[int] = 4
    dropout: float = 0.0
    multiple_of: int = 256
    ffn_multiplier: Optional[int] = None
    norm_eps: float = 1e-6
    init_std: float = 0.02
    state_dict_path: str = "./model_states_tiny_fineweb/last_model_107123.pth"

In [5]:
class InputEmbeddings(nn.Module):
    def __init__(self, embed_dim: int, vocab_size: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.vocab_size  = vocab_size
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        
    def forward(self, x):
        return self.embeddings(x)


In [6]:
class RMSNorm(nn.Module):

    def __init__(self, dim ,eps:float = 1e-6):
        super().__init__()
        self.dim = dim
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        v = x.float()
        v = v*torch.rsqrt((v**2).mean(dim=-1, keepdim=True)+self.eps)
        v = v*self.weight.float()

        return v.type_as(x)

In [7]:
class FFN(nn.Module):
    def __init__(self, dim: int, ffn_multipler: Optional[int], multiple_of: int) -> None:
        super().__init__()
        hidden_dim = 4*dim
        hidden_dim = int(2*hidden_dim/3)
        if ffn_multipler is not None:
            hidden_dim = int(hidden_dim*ffn_multipler)
        #next multiple of multiple_of
        hidden_dim = multiple_of*((hidden_dim+multiple_of-1)//multiple_of)
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)

    def forward(self,x):
        swish = F.silu(self.w1(x))
        x_V = self.w3(x)
        x = swish*x_V
        x = self.w2(x)

        return x


In [8]:
class RPositionalEmbeddings(nn.Module):
    def __init__(self, head_dim: int, seq_len: int, base: float= 10000.0):
        super().__init__()
        assert head_dim%2==0, "Head dim must be even."
        self.head_dim = head_dim
        self.seq_len = seq_len
        self.base = base

        pow = -2*(torch.arange(0, head_dim//2).float())/head_dim
        theta = (base**pow)
        m = torch.arange(seq_len).float()
        #multiply each m with all theta
        angles = torch.outer(m, theta) #seq_len,head dim/2

        full_angle = torch.concat([angles, angles], dim=1) #seq len, head dim
        self.register_buffer("cos_table", torch.cos(full_angle), persistent=False)
        self.register_buffer("sin_table", torch.sin(full_angle), persistent=False)

    def forward(self, x: torch.Tensor, positions: torch.Tensor):
        cos = self.cos_table[positions].unsqueeze(1) # batch size, 1, seq len, head dim
        sin = self.sin_table[positions].unsqueeze(1)
        # bs, heads, seq len, head dim
        out = (x*cos + torch.concat([-x[:,:,:,self.head_dim//2:], x[:,:,:,:self.head_dim//2]],\
                                     dim=-1)*sin).type_as(x)
        return out


In [9]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, dim: int, h_q: int, h_kv: int, dropout: float, rope: RPositionalEmbeddings):
        super().__init__()
        self.dim = dim
        self.h_q = h_q
        self.h_kv = h_kv
        self.repeats = h_q//h_kv
        self.positional_embed = rope
        
        self.d_k = dim//h_q
        self.w_q = nn.Linear(dim,self.d_k*h_q, bias= False)
        self.w_k = nn.Linear(dim,self.d_k*h_kv, bias= False)        
        self.w_v = nn.Linear(dim, self.d_k*h_kv, bias= False)  
        self.w_o = nn.Linear(dim, dim, bias= False)  
        self.dropout = nn.Dropout(dropout) 
        self.prev_kv = None
        
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        scores = query@key.transpose(-2,-1)/math.sqrt(d_k)

        if mask is not None:
            scores.masked_fill_(mask==0, -1e9)
        scores = scores.softmax(dim=-1)
        
        ### do we need ?
        if dropout is not None:
            scores = dropout(scores)
            
        return scores@value,scores 
        
    def forward(self, q, k, v, pos,mask):
        # batch, seq len, embed dim -> batch, seq_len, d_k*h_q
        query = self.w_q(q)
        # batch, seq len, embed dim -> batch, seq_len, d_k*h_kv
        key = self.w_k(k)
        value = self.w_v(v)
        
        # batch, h_q, seq, d_k
        query = query.view(query.shape[0], query.shape[1], self.h_q, self.d_k).transpose(1,2)
        # batch, h_kv, seq, d_k
        key = key.view(key.shape[0], key.shape[1], self.h_kv, self.d_k).transpose(1,2)
        value = value.view(value.shape[0], value.shape[1], self.h_kv, self.d_k).transpose(1,2)
        #apply positional embeddings to query and key
        query = self.positional_embed(query, pos)
        key = self.positional_embed(key, pos)

        if self.prev_kv is not None:
            pastk, pastv = self.prev_kv
            key = torch.cat([pastk, key], dim=2)
            value = torch.cat([pastv, value], dim=2)
            #print(key.shape, value.shape,query.shape)
        self.prev_kv =(key, value)
        if self.repeats!=1:
            # batch, h_q, seq, d_k
            key = key.repeat_interleave(repeats=self.repeats, dim=1)
            value = value.repeat_interleave(repeats=self.repeats, dim=1)
        #at prefill step we need causal attention
        x = F.scaled_dot_product_attention(query, key, value, attn_mask=mask,\
                                        dropout_p=self.dropout.p if self.training else 0.0,\
                                        is_causal= (query.shape[2]==key.shape[2]))
        x = x.transpose(1,2).reshape(x.shape[0],-1, self.dim)
        return self.w_o(x)
        

In [10]:
class EncoderBlock(nn.Module):
    def __init__(self, args: CFG, rope: RPositionalEmbeddings):
        super().__init__()
        self.dim = args.dim
        self.n_heads_q = args.n_heads_q
        self.n_heads_kv = args.n_heads_kv if args.n_heads_kv is not None else args.n_heads_q
        assert self.n_heads_q % self.n_heads_kv == 0, "n_heads_q must be divisible by n_heads_kv"
        self.head_dim = args.dim // self.n_heads_q
        self.attention = MultiHeadAttention(self.dim, self.n_heads_q, self.n_heads_kv, args.dropout, rope)

        self.ffn = FFN(self.dim, ffn_multipler=args.ffn_multiplier, multiple_of=args.multiple_of)
        self.norm1 = RMSNorm(self.dim, args.norm_eps)
        self.norm2 = RMSNorm(self.dim, args.norm_eps)
        self.dropout = nn.Dropout(args.dropout)

    def forward(self, x, pos, mask):
        # Attention block
        h = self.norm1(x)
        attn_out = self.attention(h, h, h, pos, mask)
        x = x + self.dropout(attn_out)

        # Feedforward block
        ffn_out = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_out)

        return x
    
    def reset_kv(self):
        self.attention.prev_kv = None

In [11]:
class Transformer(nn.Module):
    def __init__(self, args: CFG):
        super().__init__()
        self.args = args
        self.encoder_blocks = nn.ModuleList()
        rope = RPositionalEmbeddings(args.dim//args.n_heads_q, args.max_seq_len)
        for _ in range(args.n_layers):
            self.encoder_blocks.append(EncoderBlock(args, rope))
        self.embeddings = InputEmbeddings(args.dim, args.vocab_size)
        self.projection = nn.Linear(args.dim, args.vocab_size, bias=False)
        self.norm = RMSNorm(args.dim, args.norm_eps)

        self._init_weights(args.init_std)

    def forward(self, x, pos, mask):
        assert pos.max() < self.args.max_seq_len, "Position indices exceed maximum sequence length."
        x = self.embeddings(x)
        for block in self.encoder_blocks:
            x = block(x, pos, mask)
        x = self.norm(x)
        x = self.projection(x)
        return x

    def _init_weights(self, std):
        # every projection and the embedding table: normal(0, std).
        # RMSNorm gains are left at ones - matching on module type avoids them.
        for mod in self.modules():
            if isinstance(mod, nn.Linear):
                nn.init.normal_(mod.weight, mean=0.0, std=std)
                if mod.bias is not None:
                    nn.init.zeros_(mod.bias)
            elif isinstance(mod, nn.Embedding):
                nn.init.normal_(mod.weight, mean=0.0, std=std)

        # each block writes to the residual stream twice (w_o and w2), so its
        # variance would grow like sqrt(2*n_layers) with depth. Scaling the two
        # residual-writing projections by the inverse keeps it depth-independent.
        scale = 1.0/math.sqrt(2*len(self.encoder_blocks))
        for block in self.encoder_blocks:
            block.attention.w_o.weight.data.mul_(scale)
            block.ffn.w2.weight.data.mul_(scale)

    def reset_kv(self):
        for block in self.encoder_blocks:
            block.reset_kv()


In [12]:
model = Transformer(CFG())
model.reset_kv()

In [19]:
state_dict = torch.load(CFG.state_dict_path, map_location=torch.device('cpu'))
model.load_state_dict(state_dict['model_state_dict'])
model.to(device)
print(f"loaded {sum(p.numel() for p in model.parameters())/1e6:.2f}M params\n")

loaded 124.67M params



In [29]:
EOS = 1
@torch.no_grad()
def generate(prompt, n=220, temperature=0.8, top_k=50,seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    model.reset_kv()
    tokens = [EOS] if prompt==None else tokenize.encode(prompt).ids
    out = torch.tensor([tokens], device=device)
    
    #prefill kv cache first for already present tokens
    pos = torch.arange(out.size(1), device=device).repeat(1, 1)
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
        logits = model(out, pos, None)
        logits = logits[:, -1, :] / temperature
    if top_k is not None:
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float('Inf')
    probs = F.softmax(logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    out = torch.cat((out, next_token), dim=1)
    #generating next tokens
    for _ in range(min(n, CFG.max_seq_len - len(tokens))):
        ctx = out[:, -1:]
        pos = torch.tensor(out.size(1)-1, device=device).repeat(1, 1)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            logits = model(ctx, pos, None)
            logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        out = torch.cat((out, next_token), dim=1)
        if next_token.item() == EOS:
            break
    return tokenize.decode(out.squeeze().tolist())

In [30]:
prompts = [None, "There were the time when",
            "The French Revolution began in",
            "Photosynthesis is the process by which",
            "To install the package, first"]
for i, p in enumerate(prompts):
    print("="*78)
    print(f"PROMPT: {p!r}")
    print("-"*78)
    print(generate(p, seed=i))
    print()

PROMPT: None
------------------------------------------------------------------------------
When we hear a person struggling with depression, or with other mental health issues like anxiety and depression, we may feel like our own situation is not right. This is not the case. No matter how often we hear about this in the news, people may not think this is real. The reality is that depression isn’t alone. Depression is a serious illness and should be treated as such.
Depression can take a large toll on a person’s health. A lack of sleep, the lack of oxygen in their brain, and the use of alcohol or drugs all contribute towards a loss of energy.
People suffering from depression may also have physical issues like a heart attack, stroke, or getting a stroke.
Many people find that they need to talk to a doctor about the diagnosis, and that they do not want to do things that are “right” for them.

PROMPT: 'There were the time when'
-------------------------------------------------------------

In [ ]:
@torch.no_grad()
def batched_generate(prompts, n=220, temperature=0.8, top_k=50, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    model.reset_kv()
    batch_size = len(prompts)
    tokens_list = [[EOS] if prompt==None else tokenize.encode(prompt).ids for prompt in prompts]
    max_len = max(len(tokens) for tokens in tokens_list)
    out = torch.full((batch_size, n), EOS, dtype=torch.long, device=device)
    dones =torch.tensor([False] * batch_size, device=device).unsqueeze(1)
    for i, tokens in enumerate(tokens_list):
        out[i, :len(tokens)] = torch.tensor(tokens, device=device)

    for token_pos in range(1,n):
        ctx = out[:, token_pos-1:token_pos]
        pos = torch.tensor(token_pos-1, device=device).repeat(batch_size, 1)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            logits = model(ctx, pos, None)
            logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')
        probs = F.softmax(logits, dim=-1)
        next_tokens = torch.multinomial(probs, num_samples=1)
        if token_pos<max_len:
            mask_1 = out[:,token_pos] != EOS
            next_tokens[mask_1] =  out[:,token_pos:token_pos+1][mask_1]

        mask = next_tokens == EOS
        dones = dones | mask
        next_tokens[dones] = EOS

        out[:,token_pos:token_pos+1] = next_tokens
        if dones.all():
            break

    return [tokenize.decode(out[i].tolist()) for i in range(batch_size)]

In [25]:
out = batched_generate(prompts, seed=17)
for i, output in enumerate(out):
    print("="*78)
    print(f"PROMPT: {prompts[i]!r}")
    print("-"*78)
    print(output)
    print()

PROMPT: None
------------------------------------------------------------------------------
Category: Social Studies & Elementary Education
How does a classroom meet with the world's most important issues? Students will explore how the world has changed over the past three decades and how the challenges and challenges of today's technological and technological innovations have affected students' lives. In addition, they will look at how these challenges have impacted students' ability to function in an increasingly interconnected world.
This unit contains 5 units: World History, Social Studies, English and History. Each unit contains the following:
- Unit 1: World History
- Unit 2: Social Studies
- Unit 3: Social Studies
- Unit 4: Social Studies
In Unit 5, students will be introduced to the world of the past and how the challenges of today's technology are impacting students of today's society.
This unit will continue to be based on classroom lessons and will include a variety of activ